In [16]:
import pyspark
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
filepath = 'D:/ABD0008/DATA/sf-fire-calls.csv'

In [17]:
def create_sparkSession():
    spark = SparkSession.builder.appName('Fire Example').getOrCreate()
    return spark

In [18]:
def create_dataframe(spark, filepath):
    df = spark.read.csv(filepath, header = True, inferSchema = True)
    df1 = df.select('CallType', 'CallDate', 'City', 'Zipcode', 'Neighborhood', 'Delay')
    return df1

In [19]:
spark = create_sparkSession()
df = create_dataframe(spark, filepath)
#df = clean_dataset(df)
df.printSchema()

root
 |-- CallType: string (nullable = true)
 |-- CallDate: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)



In [20]:
def clean_dataset(df):
    df1 = df.withColumn('Date', to_date(col('callDate'), 'MM/dd/yyyy')).drop('callDate')
    df2 = df1.withColumn('Year',year(col('Date')))\
    .withColumn('Month', month(col('Date')))\
    .withColumn('Week', weekofyear(col('Date')))
    return df2

In [21]:
spark = create_sparkSession()
df = create_dataframe(spark, filepath)
df = clean_dataset(df)
df.printSchema()

root
 |-- CallType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Week: integer (nullable = true)



In [22]:
df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94102|          Tenderloin|1.7833333|2002-01-11|2002|    1|   2|
|Medical I

In [23]:
#create a user defined function
def mapSeason(data):
    if 2 < data < 6:
        return 'spring'
    elif 5 < data < 9:
        return 'summer'
    elif 8 < data < 12:
        return 'Autumn'
    else:
        return 'Winter'

seasonUDF = udf(mapSeason, StringType())
clean_df = df.withColumn('Season', seasonUDF(col('Month')))
clean_df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|Winter|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|Winter|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94102

In [24]:
clean_df.select('Year').groupBy('Year').count().orderBy('Year', ascending = True).show()

+----+-----+
|Year|count|
+----+-----+
|2000| 5459|
|2001| 7713|
|2002| 8090|
|2003| 8499|
|2004| 8283|
|2005| 8282|
|2006| 8174|
|2007| 8255|
|2008| 8869|
|2009| 8789|
|2010| 9341|
|2011| 9735|
|2012| 9674|
|2013|10020|
|2014|10775|
|2015|11458|
|2016|11609|
|2017|12135|
|2018|10136|
+----+-----+



In [25]:
clean_df.select('callType').where(col('Year') == 2018).distinct().show(truncate = False)

+-------------------------------+
|callType                       |
+-------------------------------+
|Elevator / Escalator Rescue    |
|Alarms                         |
|Odor (Strange / Unknown)       |
|Citizen Assist / Service Call  |
|HazMat                         |
|Vehicle Fire                   |
|Other                          |
|Outside Fire                   |
|Traffic Collision              |
|Assist Police                  |
|Gas Leak (Natural and LP Gases)|
|Water Rescue                   |
|Electrical Hazard              |
|Structure Fire                 |
|Medical Incident               |
|Fuel Spill                     |
|Smoke Investigation (Outside)  |
|Train / Rail Incident          |
|Explosion                      |
|Suspicious Package             |
+-------------------------------+



In [26]:
clean_df.select('Week').where(col('Year') == 2018).groupBy('Week').count().orderBy('count', ascending = False).collect()[0][0]

22

In [27]:
from pyspark.sql import functions as F, Window

In [28]:
clean_df.groupBy("Year", "Month").count().orderBy("Year", "Month").show()

+----+-----+-----+
|Year|Month|count|
+----+-----+-----+
|2000|    4|  335|
|2000|    5|  680|
|2000|    6|  585|
|2000|    7|  668|
|2000|    8|  678|
|2000|    9|  655|
|2000|   10|  620|
|2000|   11|  595|
|2000|   12|  643|
|2001|    1|  622|
|2001|    2|  613|
|2001|    3|  692|
|2001|    4|  636|
|2001|    5|  682|
|2001|    6|  672|
|2001|    7|  646|
|2001|    8|  660|
|2001|    9|  577|
|2001|   10|  673|
|2001|   11|  619|
+----+-----+-----+
only showing top 20 rows



In [29]:
selected_year = 2018
clean_df.filter(F.col("Year") == selected_year) \
    .groupBy("Month", "CallType").count() \
    .orderBy("Month", F.desc("count")).show()

+-----+--------------------+-----+
|Month|            CallType|count|
+-----+--------------------+-----+
|    1|    Medical Incident|  692|
|    1|              Alarms|  122|
|    1|      Structure Fire|   91|
|    1|   Traffic Collision|   42|
|    1|Citizen Assist / ...|   15|
|    1|        Outside Fire|   14|
|    1|Gas Leak (Natural...|    5|
|    1|        Water Rescue|    4|
|    1|        Vehicle Fire|    4|
|    1|               Other|    3|
|    1|   Electrical Hazard|    3|
|    1|Smoke Investigati...|    3|
|    1|Elevator / Escala...|    3|
|    1|Train / Rail Inci...|    2|
|    1|Odor (Strange / U...|    2|
|    1|          Fuel Spill|    1|
|    1|              HazMat|    1|
|    2|    Medical Incident|  635|
|    2|              Alarms|  102|
|    2|      Structure Fire|   91|
+-----+--------------------+-----+
only showing top 20 rows



In [30]:
selected_year = 2018
windowSpec = Window.partitionBy("Season").orderBy(F.desc("count"))

clean_df.filter(F.col("Year") == selected_year) \
    .groupBy("Season", "CallType").count() \
    .withColumn("rank", F.rank().over(windowSpec)) \
    .filter(F.col("rank") <= 5).drop("rank").show()

+------+-----------------+-----+
|Season|         CallType|count|
+------+-----------------+-----+
|Autumn| Medical Incident| 1514|
|Autumn|           Alarms|  251|
|Autumn|   Structure Fire|  201|
|Autumn|Traffic Collision|  100|
|Autumn|     Outside Fire|   39|
|Winter| Medical Incident| 1327|
|Winter|           Alarms|  224|
|Winter|   Structure Fire|  182|
|Winter|Traffic Collision|   79|
|Winter|     Outside Fire|   28|
|spring| Medical Incident| 2110|
|spring|           Alarms|  333|
|spring|   Structure Fire|  261|
|spring|Traffic Collision|  133|
|spring|            Other|   36|
|summer| Medical Incident| 2053|
|summer|           Alarms|  336|
|summer|   Structure Fire|  262|
|summer|Traffic Collision|  121|
|summer|     Outside Fire|   61|
+------+-----------------+-----+



In [31]:
clean_df.groupBy("CallType").pivot("Season").count().show()

+--------------------+------+------+------+------+
|            CallType|Autumn|Winter|spring|summer|
+--------------------+------+------+------+------+
|Elevator / Escala...|   105|   112|   111|   125|
|         Marine Fire|     6|     2|     5|     1|
|  Aircraft Emergency|    10|    10|     6|    10|
|Confined Space / ...|     4|     4|     4|     1|
|      Administrative|     1|     1|     1|  NULL|
|              Alarms|  4980|  4875|  4641|  4910|
|Odor (Strange / U...|   133|   123|   119|   115|
|Citizen Assist / ...|   646|   610|   599|   669|
|              HazMat|    49|    29|    26|    20|
|Watercraft in Dis...|    11|     3|     9|     5|
|           Explosion|    20|    15|    18|    36|
|           Oil Spill|     6|     4|     6|     5|
|        Vehicle Fire|   208|   219|   218|   209|
|  Suspicious Package|     5|     3|     5|     2|
|Extrication / Ent...|     5|     6|    10|     7|
|               Other|   565|   543|   531|   527|
|        Outside Fire|   461|  

In [32]:
clean_df.filter(F.col("Year") == 2018) \
    .groupBy("Month").count() \
    .orderBy(F.desc("count")).show()

+-----+-----+
|Month|count|
+-----+-----+
|   10| 1068|
|    5| 1047|
|    3| 1029|
|    8| 1021|
|    1| 1007|
|    6|  974|
|    7|  974|
|    9|  951|
|    4|  947|
|    2|  919|
|   11|  199|
+-----+-----+



In [33]:
windowYear = Window.partitionBy("Year").orderBy(F.desc("count"))

clean_df.groupBy("Year", "CallType").count() \
    .withColumn("rank", F.rank().over(windowYear)) \
    .filter(F.col("rank") == 1).drop("rank").orderBy("Year").show()

+----+----------------+-----+
|Year|        CallType|count|
+----+----------------+-----+
|2000|Medical Incident| 3408|
|2001|Medical Incident| 4653|
|2002|Medical Incident| 5046|
|2003|Medical Incident| 5056|
|2004|Medical Incident| 5137|
|2005|Medical Incident| 5084|
|2006|Medical Incident| 5027|
|2007|Medical Incident| 5114|
|2008|Medical Incident| 5692|
|2009|Medical Incident| 5671|
|2010|Medical Incident| 6186|
|2011|Medical Incident| 6413|
|2012|Medical Incident| 6296|
|2013|Medical Incident| 6690|
|2014|Medical Incident| 7176|
|2015|Medical Incident| 7812|
|2016|Medical Incident| 7999|
|2017|Medical Incident| 8330|
|2018|Medical Incident| 7004|
+----+----------------+-----+



In [34]:
clean_df.groupBy("CallType") \
    .agg(F.avg("Delay").alias("Avg_Delay")) \
    .orderBy(F.desc("Avg_Delay")).show()

+--------------------+------------------+
|            CallType|         Avg_Delay|
+--------------------+------------------+
|Mutual Aid / Assi...|38.416666311111115|
|       Assist Police|26.981903994285716|
|Train / Rail Inci...|16.452046763157895|
|      Administrative|12.261111333333332|
|              HazMat| 7.527016126612904|
|         Marine Fire| 6.928571314285715|
|Confined Space / ...| 6.915384576923078|
|Watercraft in Dis...| 6.886904817857142|
|  Suspicious Package|        6.57666672|
|   High Angle Rescue| 6.048958375000001|
|        Water Rescue| 5.507748342145695|
|               Other| 5.505155432421977|
|          Fuel Spill|  5.49222798238342|
|Citizen Assist / ...| 5.473342576604596|
|   Electrical Hazard| 5.178112038174275|
|Industrial Accidents|5.0147163340425545|
|           Oil Spill| 4.977777761904762|
|Odor (Strange / U...|       4.947959182|
|Gas Leak (Natural...| 4.583398778403141|
|Smoke Investigati...| 4.466069897851662|
+--------------------+------------

In [35]:
clean_df.groupBy("CallType") \
    .agg(F.avg("Delay").alias("Avg_Delay")) \
    .orderBy(F.desc("Avg_Delay")).limit(1).show()

+--------------------+------------------+
|            CallType|         Avg_Delay|
+--------------------+------------------+
|Mutual Aid / Assi...|38.416666311111115|
+--------------------+------------------+



In [36]:
clean_df.filter((F.col("Year") == 2018) & (F.col("City") == "SF")) \
    .groupBy("Neighborhood").count() \
    .orderBy(F.desc("count")).limit(1).show()

+------------+-----+
|Neighborhood|count|
+------------+-----+
+------------+-----+



In [37]:
clean_df.filter(F.col("Year") == 2018) \
    .groupBy("Neighborhood") \
    .agg(F.avg("Delay").alias("Avg_Delay")) \
    .orderBy(F.desc("Avg_Delay")).show()

+--------------------+------------------+
|        Neighborhood|         Avg_Delay|
+--------------------+------------------+
|           Chinatown| 6.190314097905761|
|            Presidio|5.8292270414492755|
|     Treasure Island|      5.4537037125|
|        McLaren Park| 4.744047642857142|
|Bayview Hunters P...| 4.620561956877393|
|    Presidio Heights| 4.594131472394366|
|        Inner Sunset| 4.438095199935065|
|      Inner Richmond| 4.364728682713179|
|Financial Distric...|4.3440846182901565|
|      Haight Ashbury| 4.266428599285713|
|            Seacliff| 4.261111146666666|
|  West of Twin Peaks| 4.190952390857142|
|        Potrero Hill| 4.190555557428571|
|     Pacific Heights| 4.180453718900524|
|          Tenderloin| 4.101519516597274|
|Oceanview/Merced/...| 3.947242180719424|
|           Excelsior|3.9363993797169807|
|         North Beach|  3.88924964034632|
|           Lakeshore|3.8815513650943405|
|         Mission Bay| 3.854868952191012|
+--------------------+------------

In [38]:
clean_df.groupBy("CallType").pivot("Year").avg("Delay").show()

+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|            CallType|              2000|              2001|              2002|              2003|              2004|              2005|              2006|              2007|              2008|              2009|              2010|              2011|              2012|              2013|              2014|              2015|              2016|              2017|              2018|
+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-

In [39]:
windowCityType = Window.partitionBy("Year").orderBy(F.desc("Distinct_CallTypes"))

clean_df.groupBy("Year", "City") \
    .agg(F.countDistinct("CallType").alias("Distinct_CallTypes")) \
    .withColumn("rank", F.rank().over(windowCityType)) \
    .filter(F.col("rank") == 1).drop("rank").orderBy("Year").show()

+----+-------------+------------------+
|Year|         City|Distinct_CallTypes|
+----+-------------+------------------+
|2000|           SF|                18|
|2001|           SF|                20|
|2002|           SF|                20|
|2003|           SF|                24|
|2004|           SF|                23|
|2005|           SF|                26|
|2006|           SF|                24|
|2007|           SF|                26|
|2008|           SF|                23|
|2009|           SF|                22|
|2010|           SF|                23|
|2011|           SF|                25|
|2012|           SF|                25|
|2013|           SF|                24|
|2014|San Francisco|                21|
|2014|SAN FRANCISCO|                21|
|2015|San Francisco|                24|
|2016|San Francisco|                24|
|2017|San Francisco|                26|
|2018|San Francisco|                20|
+----+-------------+------------------+



In [40]:
windowCityCalls = Window.partitionBy("Year").orderBy(F.desc("Total_Calls"))

clean_df.groupBy("Year", "City") \
    .agg(
        F.count("*").alias("Total_Calls"), 
        F.countDistinct("CallType").alias("Distinct_CallTypes")
    ) \
    .withColumn("rank", F.rank().over(windowCityCalls)) \
    .filter(F.col("rank") <= 5).drop("rank").orderBy("Year", F.desc("Total_Calls")).show()

+----+----+-----------+------------------+
|Year|City|Total_Calls|Distinct_CallTypes|
+----+----+-----------+------------------+
|2000|  SF|       5435|                18|
|2000|  TI|          8|                 6|
|2000|  YB|          5|                 1|
|2000|  HP|          3|                 2|
|2000| SFO|          3|                 1|
|2001|  SF|       7656|                20|
|2001|  TI|         37|                 6|
|2001|  YB|          6|                 2|
|2001| SFO|          4|                 1|
|2001|  DC|          3|                 2|
|2001|  HP|          3|                 1|
|2002|  SF|       8044|                20|
|2002|  TI|         18|                 5|
|2002|  PR|          7|                 3|
|2002|  HP|          6|                 3|
|2002| SFO|          5|                 1|
|2002|  DC|          5|                 2|
|2003|  SF|       8441|                24|
|2003|  TI|         33|                 3|
|2003|  PR|          9|                 4|
+----+----+

In [41]:
clean_df.groupBy("Neighborhood", "Zipcode").count().orderBy(F.desc("count")).show()

+--------------------+-------+-----+
|        Neighborhood|Zipcode|count|
+--------------------+-------+-----+
|          Tenderloin|  94102|17084|
|     South of Market|  94103|13762|
|             Mission|  94110|10444|
|Bayview Hunters P...|  94124| 9150|
|             Mission|  94103| 5445|
|          Tenderloin|  94109| 5377|
|Financial Distric...|  94105| 4235|
|      Outer Richmond|  94121| 4121|
|            Nob Hill|  94109| 3983|
| Castro/Upper Market|  94114| 3946|
|    Western Addition|  94115| 3934|
|         North Beach|  94133| 3706|
|     Sunset/Parkside|  94122| 3404|
|              Marina|  94123| 3360|
|           Excelsior|  94112| 3237|
|      Bernal Heights|  94110| 3109|
|     Sunset/Parkside|  94116| 3025|
|        Hayes Valley|  94102| 2814|
|           Lakeshore|  94132| 2658|
|       Outer Mission|  94112| 2608|
+--------------------+-------+-----+
only showing top 20 rows

